# Keyword Hierarchy Builder
Builds a multi-level keyword tree around a topic using an LLM.
Results are cached to disk so repeated queries are free and reviewable.

## 1. Imports & Setup

In [ ]:
import os
import sys
import json
import hashlib
import time
from datetime import datetime, timezone
from pathlib import Path
from datetime import datetime

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from tqdm import tqdm

sys.path.append('..')
from src.prompter import Prompter

## 2. Configuration

In [2]:
# ── LLM config ──────────────────────────────────────────────────────────────
MODEL_TYPE   = "llama4:16x17b"
CONFIG_PATH  = "../config.yaml"

# ── Query parameters ────────────────────────────────────────────────────────
ROOT_TOPIC   = "living lab"   # Starting topic
N_KEYWORDS   = 10         # Keywords requested per query
N_KEYWORDS_2 = 7         # Keywords requested per query for levels > 1
MAX_LEVELS   = 3          # How many levels deep to expand (0 = root only)

# ── Cache config ────────────────────────────────────────────────────────────
CACHE_DIR    = Path("query_cache")
CACHE_DIR.mkdir(exist_ok=True)

# ── Initialise prompter ─────────────────────────────────────────────────────
prompter = Prompter(config_path=CONFIG_PATH, model_type=MODEL_TYPE)

src.prompter - INFO - Using yiyuan host: https://yiyuan.tsc.uc3m.es
src.prompter - INFO - Using yiyuan API with host: https://yiyuan.tsc.uc3m.es


Loaded config file ../config.yaml and section logger.
Logs will be saved in data/logs
Loaded config file ../config.yaml and section llm.


## 3. Cache Helpers
Every LLM call is saved as a JSON file keyed by `topic + n_keywords`.
Re-running the notebook skips cached topics automatically.

In [ ]:
def _cache_key(topic: str, n_keywords: int) -> str:
    """Stable filename key for a (topic, n_keywords) pair."""
    raw = f"{topic.strip().lower()}_{n_keywords}"
    return hashlib.md5(raw.encode()).hexdigest()[:12] + f"__{topic.strip().lower().replace(' ', '_')}"


def cache_path(topic: str, n_keywords: int) -> Path:
    return CACHE_DIR / f"{_cache_key(topic, n_keywords)}.json"


def load_cache(topic: str, n_keywords: int) -> dict | None:
    """Return cached result dict or None if not found."""
    p = cache_path(topic, n_keywords)
    if p.exists():
        with open(p) as f:
            return json.load(f)
    return None


def save_cache(topic: str, n_keywords: int, raw_response: str, parsed: dict) -> None:
    record = {
        "topic": topic,
        "n_keywords": n_keywords,
        "timestamp": datetime.now(timezone.utc).isoformat(),  # fixed deprecation
        "raw_response": raw_response,
        "parsed": parsed,
    }
    with open(cache_path(topic, n_keywords), "w") as f:
        json.dump(record, f, indent=2)


def invalidate_cache(topic: str, n_keywords: int) -> None:
    """Delete a cached entry to force a fresh LLM call."""
    p = cache_path(topic, n_keywords)
    if p.exists():
        p.unlink()
        print(f"Cache cleared for '{topic}'.")
    else:
        print(f"No cache entry found for '{topic}'.")


def list_cache() -> pd.DataFrame:
    """Show every cached query as a DataFrame for easy review."""
    records = []
    for p in sorted(CACHE_DIR.glob("*.json")):
        with open(p) as f:
            rec = json.load(f)
        records.append({
            "file": p.name,
            "topic": rec["topic"],
            "n_keywords": rec["n_keywords"],
            "timestamp": rec["timestamp"],
            "n_parsed_keys": len(rec["parsed"]),
        })
    return pd.DataFrame(records) if records else pd.DataFrame(columns=["file","topic","n_keywords","timestamp","n_parsed_keys"])


print("Cache helpers loaded. Cache directory:", CACHE_DIR.resolve())

Cache helpers loaded. Cache directory: /export/usuarios_ml4ds/danibacaicoa/Keywords_living/Living_keywords/Notebooks/query_cache


## 4. Core Functions

In [ ]:
def build_prompt(topic: str, n_keywords: int) -> str:
    return f"""You are a JSON-only output machine. You never write explanations, greetings, or markdown.

TASK: Given a topic, return exactly {n_keywords} related keywords with specificity scores.

RULES (violations will break the system):
1. Output MUST start with {{ and end with }} — nothing before, nothing after
2. The topic itself must appear first with score 0
3. No keyword may contain the topic word "{topic}"
4. All keywords must be unique
5. Scores are integers 1-10 only (topic gets 0)
6. No markdown, no ```json, no explanations, no trailing text

SCORING:
  8-10 → direct synonyms or highly specific sub-concepts
  4-7  → broadly related fields or associated concepts  
  1-3  → loose umbrella terms

TOPIC: {topic}

OUTPUT (raw JSON only):
{{
  "{topic}": 0,
  "keyword1": score,
  "keyword2": score
}}"""




def extract_dictionary(response: str) -> dict:
    """Strip optional markdown fences and parse JSON."""
    clean = response.strip()
    if clean.startswith("```json"):
        clean = clean[7:]
    elif clean.startswith("```"):
        clean = clean[3:]
    if clean.endswith("```"):
        clean = clean[:-3]
    return json.loads(clean.strip())

def validate_keywords(parsed: dict, topic: str) -> None:
    """Raise ValueError if the parsed dict violates the expected contract."""

    if topic not in parsed:
        raise ValueError(f"Topic '{topic}' missing. Got: {list(parsed.keys())}")
    if parsed[topic] != 0:
        raise ValueError(f"Topic '{topic}' has score {parsed[topic]}, expected 0")
    if len(parsed) < 2:
        raise ValueError(f"Only {len(parsed)} entries returned, expected at least 2")

    topic_lower = topic.lower()
    for kwd, score in parsed.items():
        if kwd == topic:
            continue
        if not isinstance(score, int):
            raise ValueError(f"Score for '{kwd}' is {type(score).__name__}, expected int")
        if not (1 <= score <= 10):
            raise ValueError(f"Score {score} for '{kwd}' is out of range [1, 10]")
        # Only block if the FULL topic phrase appears inside the keyword
        if topic_lower in kwd.lower():
            raise ValueError(f"Keyword '{kwd}' contains the full topic phrase '{topic}'")



def get_keywords(topic: str, n_keywords: int, prompter,
                 use_cache: bool = True, max_retries: int = 3) -> dict:
    if use_cache:
        cached = load_cache(topic, n_keywords)
        if cached is not None:
            return cached["parsed"]

    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            prompt = build_prompt(topic, n_keywords)
            raw    = prompter.prompt(prompt)[0]
            parsed = extract_dictionary(raw)
            
            # Validate the result makes sense before accepting it
            validate_keywords(parsed, topic)
            
            save_cache(topic, n_keywords, raw, parsed)
            return parsed

        except Exception as e:
            last_error = e
            print(f"  Attempt {attempt}/{max_retries} failed for '{topic}': {e}")
            time.sleep(1 * attempt)  # simple backoff

    # Save the last bad response for inspection
    bad_path = CACHE_DIR / f"FAILED__{topic.replace(' ', '_')}.txt"
    bad_path.write_text(raw)
    raise RuntimeError(
        f"All {max_retries} attempts failed for '{topic}'. "
        f"Last error: {last_error}. Raw response saved to {bad_path}"
    )


def df_from_dict(dictionary: dict, level: int) -> pd.DataFrame:
    """
    Convert a keyword dict into a DataFrame row-set.

    - The topic entry (score == 0) is included only at level 0
      (it is already present from a parent call at deeper levels).
    - All other entries become children of the root keyword.
    """
    root_kwd = next(k for k, v in dictionary.items() if v == 0)

    rows = []
    for kwrd, score in dictionary.items():
        if score == 0:
            if level == 0:          # include root node only at the first level
                rows.append({"kwrd": kwrd, "lvl": 0, "parent": None, "score": 0})
        else:
            rows.append({"kwrd": kwrd, "lvl": level + 1, "parent": root_kwd, "score": score})

    return pd.DataFrame(rows)


print("Core functions defined.")

Core functions defined.


def build_prompt(topic: str, n_keywords: int) -> str:
    return f"""
I will give you a topic. You need to provide the {n_keywords} most related keywords to this topic.
If there are fewer than {n_keywords} valid keywords available, output only the ones that exist.
Never repeat a keyword for a given topic. All keywords must be unique.

CRITICAL RULE: None of the generated keywords may contain the topic word(s) itself.
For example, if the topic is "cancer", keywords like "cancer treatment" or "lung cancer" are
strictly forbidden. You must find distinct related terms (e.g., "oncology", "tumor", "chemotherapy").

For each keyword, assign a specificity score — a natural number between 1 and 10 inclusive.
You must also include the original topic itself in the output with a score of exactly 0.

Scoring Rubric:
  8-10 (Highly Specific)  : Direct synonyms or granular concepts exclusive to the topic.
  4-7  (Moderately Related): Broadly related fields, symptoms, or associated concepts.
  1-3  (Loosely Related)  : High-level umbrella terms or vague associations.

Topic: {topic}

Return the result as a single, flat JSON dictionary where keys are words and values are integer
scores, ordered from most specific (highest) to least specific (lowest), with the topic at the top:
{{
  "{topic}": 0,
  "keyword1": score_1,
  "keyword2": score_2
}}

Do NOT output markdown formatting, code blocks, or extra text. Output ONLY the raw JSON.
"""


def extract_dictionary(response: str) -> dict:
    """Strip optional markdown fences and parse JSON."""
    clean = response.strip()
    if clean.startswith("```json"):
        clean = clean[7:]
    elif clean.startswith("```"):
        clean = clean[3:]
    if clean.endswith("```"):
        clean = clean[:-3]
    return json.loads(clean.strip())


def get_keywords(topic: str, n_keywords: int, prompter, use_cache: bool = True) -> dict:
    """
    Query the LLM for keywords related to *topic*.
    Results are cached; set use_cache=False to force a fresh call.

    Returns the parsed keyword dict  {word: score, ...}
    """
    if use_cache:
        cached = load_cache(topic, n_keywords)
        if cached is not None:
            return cached["parsed"]

    prompt   = build_prompt(topic, n_keywords)
    raw      = prompter.prompt(prompt)[0]
    parsed   = extract_dictionary(raw)
    save_cache(topic, n_keywords, raw, parsed)
    return parsed


def df_from_dict(dictionary: dict, level: int) -> pd.DataFrame:
    """
    Convert a keyword dict into a DataFrame row-set.

    - The topic entry (score == 0) is included only at level 0
      (it is already present from a parent call at deeper levels).
    - All other entries become children of the root keyword.
    """
    root_kwd = next(k for k, v in dictionary.items() if v == 0)

    rows = []
    for kwrd, score in dictionary.items():
        if score == 0:
            if level == 0:          # include root node only at the first level
                rows.append({"kwrd": kwrd, "lvl": 0, "parent": None, "score": 0})
        else:
            rows.append({"kwrd": kwrd, "lvl": level + 1, "parent": root_kwd, "score": score})

    return pd.DataFrame(rows)


print("Core functions defined.")

## 5. Build Keyword Tree
Runs LLM queries level-by-level and accumulates results in `df`.

In [5]:
# ── Seed: root topic ────────────────────────────────────────────────────────
root_dict = get_keywords(ROOT_TOPIC, N_KEYWORDS, prompter)
df = df_from_dict(root_dict, level=0)

# ── Expand level by level ───────────────────────────────────────────────────
for lvl in range(1, MAX_LEVELS):
    level_keywords = df[df["lvl"] == lvl]["kwrd"].tolist()
    for kwd in tqdm(level_keywords, desc=f"Level {lvl} keywords"):
        kwd_dict = get_keywords(kwd, N_KEYWORDS_2, prompter)
        df_new   = df_from_dict(kwd_dict, level=lvl)
        df       = pd.concat([df, df_new], ignore_index=True)

print(f"\nDone! Tree has {len(df)} rows across {df['lvl'].max() + 1} levels.")
df

/tmp/ipykernel_7542/1335767271.py:25: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),
Level 1 keywords:  10%|█         | 1/10 [00:22<03:25, 22.84s/it]

  Attempt 1/3 failed for 'co-creation space': Keyword 'co-working space' contains the topic word 'co-creation space'
  Attempt 2/3 failed for 'co-creation space': Keyword 'co-working space' contains the topic word 'co-creation space'
  Attempt 3/3 failed for 'co-creation space': Keyword 'co-working space' contains the topic word 'co-creation space'


Level 1 keywords:  10%|█         | 1/10 [00:49<07:21, 49.04s/it]


RuntimeError: All 3 attempts failed for 'co-creation space'. Last error: Keyword 'co-working space' contains the topic word 'co-creation space'. Raw response saved to query_cache/FAILED__co-creation_space.txt

In [ ]:
kwd

'urban planning'

In [ ]:
# Debug: inspect the raw response for a failing topic
raw = prompter.prompt(build_prompt("your_failing_topic", N_KEYWORDS_2))[0]
print(repr(raw))   # shows escape chars, exact whitespace
print("---")
print(raw)         # human-readable version

'I\'m happy to help, but I need a real topic to work with. You\'ve provided "your_failing_topic" which seems to be a placeholder. Could you please provide a real topic? I\'ll be happy to assist you then.'
---
I'm happy to help, but I need a real topic to work with. You've provided "your_failing_topic" which seems to be a placeholder. Could you please provide a real topic? I'll be happy to assist you then.


## 6. Review Cache
Inspect every stored query. You can re-run a query by calling `invalidate_cache(topic, n_keywords)` and re-executing section 5.

In [ ]:
list_cache()

In [ ]:
# Inspect a specific cached response in full
REVIEW_TOPIC = ROOT_TOPIC

cached = load_cache(REVIEW_TOPIC, N_KEYWORDS)
if cached:
    print(f"Topic     : {cached['topic']}")
    print(f"Timestamp : {cached['timestamp']}")
    print(f"\nRaw LLM response:\n{cached['raw_response']}")
    print(f"\nParsed dict:")
    display(pd.DataFrame(list(cached['parsed'].items()), columns=['keyword','score']))
else:
    print("No cache entry found.")

## 7. Graph Visualisation

In [ ]:
G = nx.DiGraph()

for _, row in df.dropna(subset=["parent"]).iterrows():
    G.add_edge(row["parent"], row["kwrd"], weight=row["score"])

plt.figure(figsize=(14, 9))
pos = nx.spring_layout(G, seed=42)

# Colour nodes by level
level_map  = df.set_index("kwrd")["lvl"].to_dict()
node_colors = ["#4C9BE8" if level_map.get(n, 1) == 0
               else "#85C1E9" if level_map.get(n, 1) == 1
               else "#D6EAF8"
               for n in G.nodes()]

nx.draw(
    G, pos,
    with_labels=True,
    node_color=node_colors,
    node_size=2500,
    edge_color="gray",
    font_size=8,
    arrows=True,
)
plt.title(f"Keyword Hierarchy — root: '{ROOT_TOPIC}'")
plt.tight_layout()
plt.show()

## 8. Personalised PageRank

In [ ]:
TARGET_NODE = ROOT_TOPIC
TOP_N       = 10

if TARGET_NODE not in G.nodes():
    raise ValueError(f"Node '{TARGET_NODE}' not found in graph. Available nodes: {list(G.nodes())}")

personalization = {node: (1 if node == TARGET_NODE else 0) for node in G.nodes()}

ppr = nx.pagerank(G, alpha=0.85, personalization=personalization, weight="score")

sorted_ppr = sorted(ppr.items(), key=lambda x: x[1], reverse=True)

print(f"Top {TOP_N} keywords by Personalised PageRank (seed: '{TARGET_NODE}'):")
for node, rank in sorted_ppr[:TOP_N]:
    print(f"  {node:<25} {rank:.4f}")